In [1]:
import os
import config
import warnings
import pandas as pd

In [2]:
%config InlineBackend.figure_format = 'retina'
os.environ['PYTHONWARNINGS'] = 'ignore'
warnings.filterwarnings('ignore')
os.chdir(config.DIR_ROOT)

# Обучение модели CNN Classifier на 50 самых важных k-mer

In [3]:
# Загрузка данных
embeddings_7mer_path = os.path.join(config.DIR_INCEST_MANY, '7.csv')
embeddings_7mer = pd.read_csv(embeddings_7mer_path)
embeddings_7mer

,name,emb_0,emb_1,emb_2,emb_3,emb_4,emb_5,emb_6,emb_7,emb_8,...,emb_16374,emb_16375,emb_16376,emb_16377,emb_16378,emb_16379,emb_16380,emb_16381,emb_16382,emb_16383
0,Gypsy-5_AnMe-I_Gypsy_Anopheles_merus,0.001375,0.001146,0.000688,0.001146,0.000917,0.000917,0.000688,0.000229,0.000229,...,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0
1,Gypsy-1_AnMe-LTR_Gypsy_Anopheles_merus,0.000000,0.000000,0.000000,0.000000,0.006289,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0
2,Gypsy-35_AnFu-I_Gypsy_Anopheles_funestus,0.002708,0.000677,0.000339,0.000169,0.001354,0.001016,0.000169,0.000677,0.000677,...,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0
3,Gypsy-28_AnMe-I_Gypsy_Anopheles_merus,0.000440,0.000880,0.000220,0.000660,0.000220,0.000660,0.000440,0.000220,0.000660,...,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0
4,Gypsy-4_AnFu-I_Gypsy_Anopheles_funestus,0.000232,0.000463,0.000232,0.000463,0.000463,0.000463,0.000463,0.000232,0.000232,...,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
102253,DIRS-1F-LTR_DR_DIRS_Danio_rerio,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.0,0.0,0.0,0.001473,0.0,0.0,0.0,0.0,0.0
102254,CR1-44_DR_CR1_Danio_rerio,0.001088,0.000544,0.000000,0.001632,0.001088,0.000000,0.000544,0.000000,0.000000,...,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0
102255,DNA-4-5_DR_DNA_transposon_Danio_rerio,0.015201,0.002172,0.001086,0.000000,0.001086,0.000000,0.001086,0.001086,0.000000,...,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0
102256,hAT-27_DR_hAT_Danio_rerio,0.002107,0.000602,0.000000,0.000301,0.000602,0.000000,0.000602,0.000000,0.000000,...,0.000301,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0


In [4]:
types_7mer_path = os.path.join(config.DIR_INCEST_MANY, 'repbase_filtered.csv')
types_7mer = pd.read_csv(types_7mer_path, sep=',')
types_7mer

,name,MainType,SubType,Length,Good
0,ISL2EU-51_CGi_ISL2EU_Crassostrea_gigas,DNA transposon,IS-like/prokaryotic,2941,1
1,Kolobok-6_LMi_Kolobok_Locusta_migratoria,DNA transposon,Kolobok,1579,1
2,Transib-2N1_DTa_Transib_Drosophila_takahashii,DNA transposon,Transib,1334,1
3,Tad1-13B_BG_Tad1_Blumeria_graminis,DNA transposon,hAT/Tad1,3824,1
4,MuDR-20_TAe_MuDR_Triticum_aestivum,DNA transposon,MuDR/Mutator,4609,1
...,...,...,...,...,...
24214,L2-46_DR_L2_Danio_rerio,Non-LTR retrotransposon,LINE/L1/L2,2517,1
24215,L1Lx_II_L1_Mus_musculus,Non-LTR retrotransposon,LINE/L1/L2,6088,1
24216,LINE1-N1H_OS_L1_Oryza_sativa,Non-LTR retrotransposon,LINE/L1/L2,1253,1
24217,PteBra-2.35_L1_Pteronura_brasiliensis,Non-LTR retrotransposon,LINE/L1/L2,7031,1


In [5]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# merge по name (оставляем только те, у кого есть и эмбеддинг, и MainType)
df = types_7mer.query("Good == 1")[['name', 'MainType']].merge(
    embeddings_7mer,
    on='name',
    how='inner'
)

print("Merged:", df.shape)
print("Unique MainType:", df['MainType'].nunique())
print(df['MainType'].value_counts().head())

Merged: (24517, 16386)
Unique MainType: 3
MainType
Non-LTR retrotransposon    8193
LTR retrotransposon        8165
DNA transposon             8159
Name: count, dtype: int64


In [6]:
# признаки: все emb_*
emb_cols = [c for c in df.columns if c.startswith("emb_")]
X = df[emb_cols].to_numpy(dtype=np.float32)

# таргет: MainType -> int
le = LabelEncoder()
y = le.fit_transform(df['MainType'].astype(str))

# train/val split со стратификацией
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape, "X_val:", X_val.shape)

X_train: (19613, 16384) X_val: (4904, 16384)


In [7]:
from scripts.n12_cnn_model import CNNClassifierModel
# модель
input_dim = X_train.shape[1]
class_num = len(le.classes_)

model = CNNClassifierModel(input_dim=input_dim, class_num=class_num)
history = model.train(
    X_train, y_train,
    X_val=X_val, y_val=y_val,
    epochs=20,
    batch_size=32
)

613/613 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9938 - loss: 0.0200 - val_accuracy: 0.9156 - val_loss: 0.4349


In [9]:
# инференс + декодирование классов обратно в названия MainType
pred_classes, pred_probs = model.predict(X_val)
pred_labels = le.inverse_transform(pred_classes)

print("Примеры предсказаний:", pred_labels[:10])

Примеры предсказаний: ['DNA transposon' 'LTR retrotransposon' 'LTR retrotransposon'
 'LTR retrotransposon' 'LTR retrotransposon' 'LTR retrotransposon'
 'Non-LTR retrotransposon' 'Non-LTR retrotransposon' 'DNA transposon'
 'LTR retrotransposon']
